# 11장 레거시에서 클린으로: 유지 보수하기 위한 파이썬 리팩토링

파이썬으로 구현하는 클린 아키텍처 - 11장 레거시에서 클린으로: 유지 보수하기 위한 파이썬 리팩토링 코드 예제

> **[노트북 참고]** 아래 셀은 노트북 환경에서 `EcomApp` 코드를 import할 수 있도록 경로를 설정합니다. 반드시 첫 번째로 실행해 주세요.

In [ ]:
# ============================================================
# [추가] 노트북 환경 설정 - EcomApp 코드 import를 위한 경로 구성
# Google Colab: 깃허브에서 레포 클론 후 경로 설정
# 로컬 환경: 현재 디렉토리의 EcomApp 폴더 경로 설정
# 반드시 첫 번째로 실행 필요
# ============================================================
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/songys/Clean-Architecture-with-Python.git /content/repo
    ECOMAPP_PATH = '/content/repo/Chapter_11/EcomApp'
else:
    ECOMAPP_PATH = os.path.join(os.getcwd(), 'EcomApp')

if ECOMAPP_PATH not in sys.path:
    sys.path.insert(0, ECOMAPP_PATH)

## 개요

이 장에서는 클린 아키텍처로의 변환 과정을 통해, 기존 시스템의 비즈니스 가치를 유지하면서 체계적으로 개선하는 방법을 알아본다.

이 장에서 다루는 주요 주제:
* 아키텍처 변환 평가 및 계획 수립
* 점진적 클린 아키텍처 구현

### 00_legacy_app.py

## 레거시 앱 분석

수년에 걸쳐 발전해 온 주문 처리 시스템으로, 단순한 Flask 앱에서 결제 처리와 주문 이행 기능이 추가되었다. 7가지 관심사가 하나의 라우트에 혼합된 전형적 안티패턴을 보여준다.

In [ ]:
# 레거시 앱 분석: 모든 관심사가 하나의 라우트 핸들러에 혼합된 전형적 안티패턴
# 문제점: 입력 검증, DB 접근, 비즈니스 로직, 외부 API 호출, HTTP 응답이 단일 함수에 존재
# → 테스트 어려움, 변경 시 영향 범위 예측 불가, 재사용 불가
# Colab에서는 flask/requests 미설치 시 스텁으로 코드 구조 학습 가능
# order_system/app.py
# [수정] flask/requests 미설치 시에도 동작하도록 보호
try:
    from flask import Flask, request, jsonify
except ImportError:
    from types import SimpleNamespace
    class Flask:
        def __init__(self, *a, **kw):
            self.config = {}
        def route(self, *a, **kw):
            def decorator(f): return f
            return decorator
    request = SimpleNamespace(get_json=lambda: {}, args={}, form={})
    def jsonify(data=None, **kwargs): return data or kwargs

import sqlite3

try:
    import requests
except ImportError:
    class _StubResponse:
        status_code = 200
        def json(self): return {}
    class requests:  # type: ignore
        @staticmethod
        def post(*a, **kw): return _StubResponse()

app = Flask(__name__)


def get_db_connection():
    """DB 연결 - 라우트에서 직접 호출하는 안티패턴"""
    conn = sqlite3.connect("orders.db")
    conn.row_factory = sqlite3.Row
    return conn


@app.route("/orders", methods=["POST"])
def create_order():
    """안티패턴: 7가지 관심사가 하나의 함수에 혼합"""
    data = request.get_json()

    # 관심사 1: 입력 검증 (인터페이스 계층의 역할)
    if not data or not "customer_id" in data or not "items" in data:
        return jsonify({"error": "필수 필드 누락"}), 400

    # 관심사 2: DB 직접 접근 (인프라 계층의 역할)
    conn = get_db_connection()

    # 관심사 3+4: 비즈니스 로직(재고 확인) + 데이터 접근 혼합
    total_price = 0
    for item in data["items"]:
        product = conn.execute(
            "SELECT * FROM products WHERE id = ?", (item["product_id"],)
        ).fetchone()
        if not product or product["stock"] < item["quantity"]:
            conn.close()
            return jsonify({"error": f'제품 {item["product_id"]}의 재고가 없음'}), 400

        # 관심사 5: 가격 계산 (도메인 계층의 역할)
        price = product["price"] * item["quantity"]
        total_price += price

    # 관심사 6: 외부 결제 서비스 직접 호출 (인프라 계층의 역할)
    payment_result = requests.post(
        "https://payment-gateway.example.com/process",
        json={"customer_id": data["customer_id"], "amount": total_price, "currency": "USD"},
    )

    if payment_result.status_code != 200:
        conn.close()
        return jsonify({"error": "결제 실패"}), 400

    # 관심사 7: 직접 SQL로 주문/항목 생성 및 재고 업데이트
    order_id = conn.execute(
        "INSERT INTO orders (customer_id, total_price, status) VALUES (?, ?, ?)",
        (data["customer_id"], total_price, "PAID"),
    ).lastrowid

    for item in data["items"]:
        conn.execute(
            "INSERT INTO order_items (order_id, product_id, quantity, price) VALUES (?, ?, ?, ?)",
            (order_id, item["product_id"], item["quantity"], price),
        )
        conn.execute(
            "UPDATE products SET stock = stock - ? WHERE id = ?",
            (item["quantity"], item["product_id"]),
        )
    conn.commit()
    conn.close()
    return jsonify({"order_id": order_id, "status": "success"}), 201

### 01_order_entity.py

## 1단계: Order 엔터티 정의

도메인 계층부터 시작하여 안정적인 핵심을 마련한다. `OrderStatus` 열거형으로 문자열 상수를 대체하고, `OrderItem` 값 객체로 주문-제품 관계를 표현하며, `Order` 엔터티에 상태 전이 규칙과 가격 계산 로직을 캡슐화한다.

In [ ]:
# 1단계 - Order 엔티티 정의: 레거시 코드에서 추출한 순수 도메인 모델
# 이전에 코드베이스 전체에 흩어져 있던 비즈니스 규칙을 엔티티에 캡슐화
# 외부 의존성(DB, HTTP) 없이 비즈니스 규칙 테스트 가능
# Colab에서 실행 시 외부 패키지 불필요 (Python 표준 라이브러리만 사용)
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import List, Optional
from uuid import UUID, uuid4


# 주문 상태 열거형: 문자열 상수를 대체하여 타입 안전성 확보
class OrderStatus(Enum):
    CREATED = "CREATED"       # 주문 생성됨
    PAID = "PAID"             # 결제 완료
    FULFILLING = "FULFILLING" # 이행 중
    SHIPPED = "SHIPPED"       # 배송됨
    DELIVERED = "DELIVERED"   # 배달 완료
    CANCELED = "CANCELED"     # 취소됨


# 주문 항목 값 객체: 주문과 제품 간 관계를 표현하는 불변 객체
@dataclass
class OrderItem:
    product_id: UUID
    quantity: int
    price: float

    @property
    def total_price(self) -> float:
        """항목별 총 가격 계산 - 비즈니스 로직을 값 객체에 캡슐화"""
        return self.price * self.quantity


# 주문 엔티티: 핵심 비즈니스 규칙(상태 전이, 가격 계산)을 캡슐화
@dataclass
class Order:
    customer_id: UUID
    items: List[OrderItem] = field(default_factory=list)
    id: UUID = field(default_factory=uuid4)
    status: OrderStatus = OrderStatus.CREATED  # 초기 상태: CREATED
    created_at: datetime = field(default_factory=lambda: datetime.now())
    updated_at: Optional[datetime] = None

    @property
    def total_price(self) -> float:
        """주문 총 가격 - 모든 항목의 합계"""
        return sum(item.total_price for item in self.items)

    def add_item(self, item: OrderItem) -> None:
        """항목 추가 - 변경 시간 자동 갱신"""
        self.items.append(item)
        self.updated_at = datetime.now()

    def mark_as_paid(self) -> None:
        """결제 완료 표시 - 상태 전이 규칙 강제 (CREATED → PAID만 허용)"""
        if self.status != OrderStatus.CREATED:
            raise ValueError(f"결제 완료로 표시할 수 없음: 주문 상태가 {self.status.value}")
        self.status = OrderStatus.PAID
        self.updated_at = datetime.now()

### 02_product_entity.py

## 1단계: Product 엔터티 정의

이제 주문 엔터티는 이전에 코드베이스 전체에 흩어져 있던 핵심 비즈니스 개념을 적절히 캡슐화한다. 주문 상태를 '결제 완료'로 표시할 때 상태 변환을 검증하는 등 비즈니스 규칙을 강제하는 메서드를 구현했다.

In [ ]:
# 1단계 - Product 엔티티 정의: 재고 관리 로직을 도메인 엔티티에 캡슐화
# "묻지 말고 시켜라(Tell, Don't Ask)" 원칙 적용
# 이전: 컨트롤러에서 재고 확인 후 직접 SQL UPDATE
# 이후: product.decrease_stock()으로 비즈니스 규칙 강제 (음수 재고 방지)
from dataclasses import dataclass, field
from uuid import UUID, uuid4


@dataclass
class Product:
    """제품 엔티티 - 재고 관리 비즈니스 규칙 캡슐화"""
    name: str
    price: float
    stock: int
    id: UUID = field(default_factory=uuid4)

    def decrease_stock(self, quantity: int) -> None:
        """재고 감소 - 비즈니스 규칙 강제 (양수 확인, 재고 부족 방지)"""
        if quantity <= 0:
            raise ValueError("수량은 양수여야 합니다")
        if quantity > self.stock:
            raise ValueError(f"재고가 부족합니다: 요청 수량 {quantity}, 가용 수량 {self.stock}")
        self.stock -= quantity

### 03_order_repository.py

## 1단계: OrderRepository 인터페이스

도메인 계층이 필요로 하는 작업을 구현 방식 없이 정의하는 추상 리포지토리이다. 5장의 `TaskRepository`와 동일한 패턴으로, 도메인 계층이 특정 영속성 메커니즘(SQLite, PostgreSQL 등)으로부터 독립성을 유지하게 한다.

In [ ]:
# 1단계 - OrderRepository 인터페이스: 도메인 계층에 위치하는 추상 리포지토리
# 구현 방식(SQLite, PostgreSQL, 인메모리 등)을 명시하지 않고 필요한 작업만 정의
# 5장의 TaskRepository와 동일한 패턴 - 의존성 역전 원칙(DIP) 적용
# Colab에서 실행 시 외부 패키지 불필요 (abc는 Python 표준 라이브러리)
from abc import ABC, abstractmethod
from order_system.domain.entities.order import Order
from typing import List, Optional
from uuid import UUID


class OrderRepository(ABC):
    """주문 리포지토리 추상 인터페이스 - 도메인 계층의 포트"""

    @abstractmethod
    def save(self, order: Order) -> None:
        """주문 저장 - 구체적 저장 방식은 구현체에서 결정"""
        pass

    @abstractmethod
    def get_by_id(self, order_id: UUID) -> Optional[Order]:
        """ID로 주문 조회"""
        pass

    @abstractmethod
    def get_by_customer(self, customer_id: UUID) -> List[Order]:
        """고객의 모든 주문 조회"""
        pass

### 04_order_service.py

## 1단계: PaymentService 인터페이스

레거시 코드에서 `requests.post()`로 직접 호출하던 결제 로직을 추상 인터페이스로 분리한다. 도메인 엔터티를 DB/프레임워크 없이 독립 테스트할 수 있고, 비즈니스 규칙이 도메인 모델에 중앙화된다.

In [ ]:
# 1단계 - PaymentService 인터페이스: 외부 결제 서비스를 위한 추상 인터페이스
# 레거시: 라우트에서 requests.post()로 결제 API 직접 호출
# 클린: PaymentService 인터페이스를 정의하여 구현체(실제 API, Mock 등)와 분리
# PaymentResult 값 객체로 결제 결과를 명확하게 표현
# order_system/domain/services/payment_service.py
from order_system.domain.entities.order import Order
from dataclasses import dataclass
from abc import ABC, abstractmethod
from typing import Optional


# 결제 결과 값 객체 - 성공/실패와 오류 메시지를 캡슐화
@dataclass
class PaymentResult:
    success: bool
    error_message: Optional[str] = None


# 결제 서비스 추상 인터페이스 - 도메인 계층의 포트
class PaymentService(ABC):
    @abstractmethod
    def process_payment(self, order: Order) -> PaymentResult:
        """주문에 대한 결제 처리 - 구체적 결제 방식은 구현체에서 결정"""
        pass

### 05_create_order_route.py

## 2단계: 변환 경계 식별

레거시 코드에서 클린 인터페이스를 도입할 수 있는 **자연스러운 경계**를 식별한다. 주문 생성 프로세스는 입력 검증, 재고 확인, 결제 처리, 주문 저장이라는 명확한 단계가 있어 변환의 좋은 출발점이 된다. 코드를 수정하기 전에 **회귀 테스트를 먼저 구축**하여 기존 동작의 안전망을 확보해야 한다.

In [ ]:
# 2단계 - 변환 경계 식별: 레거시 코드의 각 책임을 클린 아키텍처 계층에 매핑
# 코드 수정 전에 회귀 테스트를 먼저 구축하여 기존 동작의 안전망 확보
#
# 레거시 create_order() 라우트 핸들러의 7가지 책임:
# 1. 입력 검증 (HTTP 요청 파싱)
# 2. 재고 확인 (직접 DB 쿼리)
# 3. 가격 계산 (비즈니스 로직)
# 4. 결제 처리 (외부 API 호출)
# 5. 주문 저장 (직접 SQL INSERT)
# 6. 재고 업데이트 (직접 SQL UPDATE)
# 7. HTTP 응답 생성
#
# 클린 아키텍처 계층별 매핑:
# ┌─────────────────────────────────────────────────────┐
# │ 인터페이스 어댑터: 입력 검증, HTTP 응답 (1, 7)       │
# │ 애플리케이션:     재고 확인, 가격 계산, 결제 조율 (2,3,4) │
# │ 도메인:          주문 생성, 상태 변경 (5)             │
# │ 인프라:          DB 쿼리, 외부 API (2, 4, 5, 6)      │
# └─────────────────────────────────────────────────────┘
#
# 핵심 원칙: 도메인 계층부터 시작하여 바깥으로 확장

### 06_create_order_tests.py

## 2단계: 회귀 테스트 구축

컨트롤러 메서드는 명확한 입력과 출력=이 있는 독립적인 워크플로에 해당하므로 초기 변환에 이상적인 후보이다. 코드를 수정하기 전에 현재 동작을 검증하는 포착하는 포괄적인 테스트 커버리지를 구축해야 한다.

In [ ]:
# 2단계 - 회귀 테스트 구축: 레거시 코드 변환 전 기존 동작의 안전망 확보
# 코드를 수정하기 전에 현재 동작을 검증하는 테스트를 먼저 작성
# → 리팩토링 과정에서 기존 기능이 깨지지 않았는지 확인 가능
# Colab에서는 DB/Flask 의존성 때문에 직접 실행 불가 (코드 구조 학습용)
# test_order_creation.py
def test_create_order_success():
    """주문 생성 성공 시나리오 - 레거시 동작 검증"""
    # 테스트 데이터 및 예상 결과 설정
    response = client.post(
        "/orders", json={"customer_id": "12345", "items": [{"product_id": "789", "quantity": 2}]}
    )

    # 검증 1: HTTP 상태 코드와 응답 구조 확인
    assert response.status_code == 201
    assert "order_id" in response.json

    # 검증 2: 데이터베이스 상태 확인 - 주문이 올바른 값으로 저장되었는지
    conn = get_db_connection()
    order = conn.execute(
        "SELECT * FROM orders WHERE id = ?", (response.json["order_id"],)
    ).fetchone()
    assert order["status"] == "PAID"  # 결제 완료 상태 확인


# 추가 시나리오: 재고 부족, 결제 실패, 필수 필드 누락 등

### 07_sqlite_order_repository.py

## 2단계: SQLite 리포지토리 어댑터

테스트가 마련되면, 클린 도메인 모델과 기존 인프라를 연결할 인터페이스 계층 구성 요소 구현을 시작할 수 있다. 리포지토리 어댑터 구현 첫 번째 단계는 기존 데이터베이스 스키마와 상호작용하면서 클린 도메인 인터페이스를 충족하는 리포지토리 어댑터를 생성하는 것이다.

In [ ]:
# 2단계 - SQLite 리포지토리 어댑터: 클린 도메인 인터페이스와 기존 DB 연결
# OrderRepository 추상 인터페이스를 구현하여 기존 SQLite 스키마와 상호작용
# 6장의 FileTaskRepository와 동일한 패턴 - 인프라 계층에 위치
# Colab에서는 SQLite가 Python에 내장되어 있으므로 동일하게 동작
# order_system/infrastructure/repositories/sqlite_order_repository.py

# [추가] 노트북 환경에서 RepositoryError가 정의되어 있지 않으므로 간단히 정의
class RepositoryError(Exception):
    """리포지토리 계층의 커스텀 예외 - 인프라 오류를 도메인 독립적으로 전달"""
    pass

class SQLiteOrderRepository(OrderRepository):
    """SQLite 기반 주문 리포지토리 구현체 - 인프라 계층"""

    # ... 구현 생략

    def save(self, order: Order) -> None:
        """주문 저장 - 기존 SQLite 스키마에 맞게 삽입/업데이트"""
        conn = sqlite3.connect(self.db_path)

        try:
            cursor = conn.cursor()
            # 주문 존재 여부에 따라 삽입 또는 업데이트 분기
            if self._order_exists(conn, order.id):
                pass  # [보완] ... SQL 업데이트 작업 ...
            else:
                pass  # [보완] ... SQL 삽입 작업 ...

                # ... 주문 항목에 대한 SQL 작업 ...
            conn.commit()
        except Exception as e:
            conn.rollback()  # 오류 시 트랜잭션 롤백
            raise RepositoryError(f"주문 저장 실패: {str(e)}")
        finally:
            conn.close()  # 반드시 연결 해제

> **[추가]** 아래 셀은 `CreateOrderUseCase`에서 사용되는 `ProductRepository` 인터페이스를 노트북 환경에서 사용할 수 있도록 EcomApp에서 import합니다.

In [ ]:
# [추가] ProductRepository 인터페이스 import
# CreateOrderUseCase가 ProductRepository에 의존하므로 미리 import
# Colab에서는 EcomApp 경로 설정이 완료된 후 실행 필요
from order_system.domain.repositories.product_repository import ProductRepository

### 08_create_order_use_case.py

## 2단계: 주문 생성 유스 케이스

이 리포지토리 어댑터는 변환 전략에서 핵심적인 역할을 수행한다. 6장에서 작업 관리 시스템을 위해 유사한 리포지토리 구현체를 소개한 적이 있다.

In [ ]:
# 2단계 - 주문 생성 유스케이스: 레거시 라우트의 비즈니스 로직을 애플리케이션 계층으로 분리
# 이전: 라우트 핸들러에서 재고 확인, 결제, 저장을 직접 수행
# 이후: 유스케이스가 추상 인터페이스(리포지토리, 결제 서비스)를 통해 조율
# → 프레임워크 없이도 단위 테스트 가능
# order_system/application/use_cases/create_order.py
from dataclasses import dataclass
from typing import List, Dict, Any
from uuid import UUID


# 요청 DTO: 웹/CLI 등 인터페이스에서 전달받는 주문 생성 요청 데이터
@dataclass
class CreateOrderRequest:
    customer_id: UUID
    items: List[Dict[str, Any]]


# 주문 생성 유스케이스: 비즈니스 흐름 조율 (의존성 주입을 통한 느슨한 결합)
@dataclass
class CreateOrderUseCase:
    order_repository: OrderRepository      # 추상 리포지토리 (구현체: SQLite, 인메모리 등)
    product_repository: ProductRepository  # 제품 리포지토리
    payment_service: PaymentService        # 결제 서비스 (구현체: 실제 API, Mock 등)

    def execute(self, request: CreateOrderRequest) -> Order:
        """주문 생성 흐름: 항목 추가 → 재고 확인 → 결제 처리 → 저장"""
        # 1. 기본 정보로 주문 엔티티 생성
        order = Order(customer_id=request.customer_id)

        # 2. 주문에 항목 추가 및 재고 확인
        for item_data in request.items:
            product_id = UUID(item_data["product_id"])
            quantity = item_data["quantity"]

            # [보완] 재고 검증 - 추상 리포지토리를 통한 제품 조회
            product = self.product_repository.get_by_id(product_id)
            if not product:
                raise ValueError(f"ID가 {product_id}인 제품을 찾을 수 없습니다")

            # 도메인 엔티티의 비즈니스 규칙으로 재고 감소 (이전: 직접 SQL UPDATE)
            product.decrease_stock(quantity)
            self.product_repository.update(product)

        # 3. 결제 처리 - 추상 서비스를 통한 호출 (이전: requests.post() 직접 호출)
        payment_result = self.payment_service.process_payment(order)
        if not payment_result.success:
            raise ValueError(f"결제에 실패했습니다: {payment_result.error_message}")

        # 4. 주문을 결제 완료로 표시하고 저장 (도메인 규칙 강제)
        order.mark_as_paid()
        self.order_repository.save(order)

        return order

### 09_order_controller.py

## 2단계: 주문 컨트롤러

execute 메서드의 후반부는 결제 처리, 주문 상태 업데이트, 완료된 주문 저장을 처리하며 주문 생성 프로세스를 계속한다. 이 유스 케이스는 클린 아키텍처의 관심사 분리를 실제로 적용한 예시이다.

In [ ]:
# 2단계 - 주문 컨트롤러: 인터페이스 어댑터 계층의 형식 변환 역할
# 웹 요청(JSON)을 도메인 요청(CreateOrderRequest)으로 변환
# 도메인 응답(Order)을 웹 응답(dict)으로 변환
# 컨트롤러 자체는 프레임워크(Flask)에 의존하지 않음
# order_system/interfaces/controllers/order_controller.py
from dataclasses import dataclass
from typing import Dict, Any
from uuid import UUID

@dataclass
class OrderController:
    """주문 컨트롤러 - 외부 형식과 도메인 형식 간 변환 담당"""
    create_use_case: CreateOrderUseCase  # 유스케이스 의존성 주입

    def handle_create_order(self, request_data: Dict[str, Any]) -> Dict[str, Any]:
        """웹 요청 처리: JSON → 도메인 요청 → 유스케이스 실행 → 웹 응답"""
        try:
            # 입력 변환: 웹 형식(문자열)을 도메인 형식(UUID)으로 변환
            customer_id = UUID(request_data['customer_id'])
            items = request_data['items']

            request = CreateOrderRequest(
                customer_id=customer_id,
                items=items
            )

            # 유스케이스 실행 - 비즈니스 로직은 유스케이스에 위임
            order = self.create_use_case.execute(request)

            # 출력 변환: 도메인 응답(Order 엔티티)을 웹 형식(dict)으로 변환
            return {
                'order_id': str(order.id),
                'status': order.status.value
            }
        except ValueError as e:
            raise  # [보완] 예외는 라우트 핸들러에서 HTTP 응답으로 변환

### 10_create_order_route_refactor.py

## 3단계: 스트랭글러 피그(Strangler Fig) 패턴 적용

이 파일의 시작 부분만으로도 여러 아키텍처 문제가 드러난다. 라우트 핸들러는 SQLite와 requests를 직접 임포트하여 이런 특정 구현체들에 대한 강한 의존성을 설정한다.

In [ ]:
# 3단계 - 스트랭글러 피그(Strangler Fig) 패턴: 기능 플래그로 점진적 전환
# 레거시 코드와 클린 구현을 공존시켜 안전하게 전환
# USE_CLEAN_ARCHITECTURE 플래그로 트래픽을 클린 구현으로 점진적 이동
# → 문제 발생 시 플래그만 끄면 즉시 레거시로 복구 가능
# Colab에서는 Flask 스텁 사용 시 코드 구조 학습용으로 참고
# order_system/app.py의 수정된 라우트
# [수정] flask 미설치 시에도 동작하도록 보호
try:
    from flask import request, jsonify
except ImportError:
    pass  # [추가] 이전 셀에서 이미 스텁 정의됨

@app.route('/orders', methods=['POST'])
def create_order():
    """스트랭글러 피그 패턴: 기능 플래그로 레거시/클린 구현 전환"""
    data = request.get_json()

    # 기본 입력 검증은 라우트 핸들러에 유지 (인터페이스 계층의 역할)
    if not data or not 'customer_id' in data or not 'items' in data:
        return jsonify({'error': '필수 필드가 누락되었습니다'}), 400

    try:
        # 기능 플래그: 클린 아키텍처 구현 사용 여부 제어
        if app.config.get('USE_CLEAN_ARCHITECTURE', False):
            # 클린 구현: 컨트롤러 → 유스케이스 → 도메인 엔티티 흐름
            result = order_controller.handle_create_order(data)
            return jsonify(result), 201
        else:
            pass  # [보완] 기존 레거시 구현은 여기에 유지 ...
    except ValueError as e:  # [수정] ValidationError → ValueError
        return jsonify({'error': str(e)}), 400
    except SystemError:
        return jsonify({'error': '내부 서버 오류'}), 500